<a href="https://colab.research.google.com/github/shahbazmaryam25/Data-Science-and-Generative-AI/blob/main/week_4_4_01_statistics_vs_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# package / library for the traditional implementation
import statsmodels.api as sm

# package / library for the ML implementation
from sklearn.linear_model import Lasso

# error metrics to assess the model
from sklearn.metrics import r2_score

df = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/housing.csv", header=None)

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [2]:
x_values = df.drop([13], axis = 1)
print(f'X values: \n {x_values.head()}\n')
print('\n')

y_value = df[13]
print(f'Y value: \n {y_value[0:5]}')

X values: 
         0     1     2   3      4      5     6       7   8      9     10  \
0  0.00632  18.0  2.31   0  0.538  6.575  65.2  4.0900   1  296.0  15.3   
1  0.02731   0.0  7.07   0  0.469  6.421  78.9  4.9671   2  242.0  17.8   
2  0.02729   0.0  7.07   0  0.469  7.185  61.1  4.9671   2  242.0  17.8   
3  0.03237   0.0  2.18   0  0.458  6.998  45.8  6.0622   3  222.0  18.7   
4  0.06905   0.0  2.18   0  0.458  7.147  54.2  6.0622   3  222.0  18.7   

       11    12  
0  396.90  4.98  
1  396.90  9.14  
2  392.83  4.03  
3  394.63  2.94  
4  396.90  5.33  



Y value: 
 0    24.0
1    21.6
2    34.7
3    33.4
4    36.2
Name: 13, dtype: float64


1. What is linear regression?
- Used to predict a continuous numerical target (\(y\)), such as a price or sales figure.
- Assumes the relationship between the independent variables (\(x\)) and the target is approximately linear.
2. The model
- \(y\) = dependent variable / target.
- \(b_0\) = intercept.
- \(b1,...,b13) = coefficients (weights) for the 13 independent variables.
- Each coefficient represents the contribution of its corresponding variable to the prediction, holding the other variables constant.
3. How does the model learn?

Linear regression uses Ordinary Least Squares (OLS) to find the coefficients that minimise the sum of squared errors.

Ordinary Least Squares (OLS) finds the best-fitting regression line by minimising the sum of squared errors.



Residual/error = Actual value − Predicted value.



Errors can be positive or negative, so adding them directly could cancel them out.



Squaring each error makes it positive, preventing cancellation.



OLS adds all the squared errors together and chooses the coefficients that make this total as small as possible.

In [3]:
mod = sm.OLS(y_value, x_values)
res = mod.fit()
print(res.summary())

                                 OLS Regression Results                                
Dep. Variable:                     13   R-squared (uncentered):                   0.959
Model:                            OLS   Adj. R-squared (uncentered):              0.958
Method:                 Least Squares   F-statistic:                              891.3
Date:                Wed, 28 Oct 2026   Prob (F-statistic):                        0.00
Time:                        19:49:02   Log-Likelihood:                         -1523.8
No. Observations:                 506   AIC:                                      3074.
Df Residuals:                     493   BIC:                                      3128.
Df Model:                          13                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

1. Goodness of fit: The model has an \(R^2\) of 96%, suggesting a strong fit to the observed data. However, this is an uncentred \(R^2\) because the model has no intercept, so it should be interpreted cautiously.

2. Hypothesis testing: The overall F-statistic and individual p-values help assess statistical significance. Most independent variables have \(p<0.05\), but variables 3, 5 and 7 do not. These could be tested for removal and the model reassessed.

3. Other diagnostics: The correlation matrix suggests multicollinearity (independent variables are strongly correlated). Skewness, kurtosis and other diagnostics should also be examined.

4. Limitations: A high \(R^2\) alone does not prove the model is correct. The large condition number also warns of possible multicollinearity or numerical problems.

Key takeaway: The initial statistical regression model fits the data well, but it needs further checks. Next, the model will be approached from a machine-learning perspective.

In [4]:

# split data into training and test
from sklearn.model_selection  import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(x_values, y_value, test_size = 0.2, random_state=1234)

# print the shapes to check everything is OK
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(404, 13)
(102, 13)
(404,)
(102,)


1. What is regularisation?
- A technique used to reduce overfitting and help a model generalise to new data.
- It adds a penalty to the model's ordinary least squares (OLS) loss function.
2. What is LASSO regression?
- LASSO = Least Absolute Shrinkage and Selection Operator.
- It adds a penalty based on the sum of the absolute values of the coefficients (weights) to the OLS objective.

Why use LASSO?
- Helps reduce overfitting.
- Can reduce model variance by limiting large coefficients.
- Can shrink some coefficients exactly to zero, effectively selecting features.


In [5]:
# configure the model
l1_algo = Lasso(alpha=0.25)
l1_algo

Lasso(alpha=0.25)